# Curvilinear Boundary Conditions

Connect ghost zones, parity, and radiation stencils to curvilinear generated-code boundary conditions.

Navigation: [Index](../index.ipynb) | Previous: [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb) | Next: [GeneralRFM and Fisheye Coordinates](generalrfm_and_fisheye.ipynb)

## Why This Matters
Curvilinear grids have outer boundaries, inner ghost zones, and coordinate singularities. Parity tells ghost-zone fills how tensor components change sign.

## What You Will See
You will inspect printed expressions, generated artifacts, or diagnostic tables and then connect them to the next notebook.

Prerequisite bridge: this notebook follows [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb). Terms are defined here before they are used.

## Parity and Radiation Stencils
Parity classes tell an inner ghost-zone fill whether a tensor component keeps or flips sign. Outer radiation conditions use one-sided finite-difference coefficients.

In [1]:
import nrpy.c_function as cfc
import nrpy.params as par
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import apply_bcs_outerradiation_and_inner as outer_bcs
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import bcstruct_set_up, register_all

par.set_parval_from_str("Infrastructure", "BHaH")
par.set_parval_from_str("fp_type", "double")
parity_classes = [
    "scalar",
    "x0 vector or one-form component",
    "x1 vector or one-form component",
    "x2 vector or one-form component",
    "x0x0 symmetric rank-2 component",
    "x0x1 symmetric rank-2 component",
    "x0x2 symmetric rank-2 component",
    "x1x1 symmetric rank-2 component",
    "x1x2 symmetric rank-2 component",
    "x2x2 symmetric rank-2 component",
]
print("parity classes:")
for index, name in enumerate(parity_classes):
    print(index, name)

parity_code = bcstruct_set_up.parity_conditions_symbolic_dot_products("Spherical")
assignment_block = parity_code[parity_code.index("{"):]
print("complete Spherical parity assignment block:")
print(assignment_block)

coeffs, offsets = outer_bcs.get_arb_offset_FD_coeffs_indices(4, -1, 1)
print("fourth-order offset derivative coefficients:")
for coeff, offset in zip(coeffs, offsets):
    print(coeff, offset)

for name in list(cfc.CFunction_dict):
    if "bc" in name.lower() or "boundary" in name.lower() or "inbounds" in name.lower() or "parity" in name.lower():
        cfc.CFunction_dict.pop(name, None)
register_all.register_C_functions({"Spherical"}, radiation_BC_fd_order=4)
print("registered boundary functions:")
for name in sorted(key for key in cfc.CFunction_dict if "bc" in key.lower() or "boundary" in key.lower() or "inbounds" in key.lower() or "parity" in key.lower()):
    print(name)
    print(cfc.CFunction_dict[name].function_prototype.splitlines()[0])

parity classes:
0 scalar
1 x0 vector or one-form component
2 x1 vector or one-form component
3 x2 vector or one-form component
4 x0x0 symmetric rank-2 component
5 x0x1 symmetric rank-2 component
6 x0x2 symmetric rank-2 component
7 x1x1 symmetric rank-2 component
8 x1x2 symmetric rank-2 component
9 x2x2 symmetric rank-2 component
Setting up reference_metric[Spherical]...


complete Spherical parity assignment block:
{
const REAL tmp0 = cos(xx1)*cos(xx1_inbounds);
const REAL tmp1 = sin(xx1)*sin(xx1_inbounds);
const REAL tmp2 = sin(xx2)*sin(xx2_inbounds);
const REAL tmp3 = cos(xx2)*cos(xx2_inbounds);
const REAL tmp4 = tmp0 + tmp1*tmp2 + tmp1*tmp3;
const REAL tmp5 = tmp0*tmp2 + tmp0*tmp3 + tmp1;
const REAL tmp6 = tmp2 + tmp3;
REAL_parity_array[0] = 1;
REAL_parity_array[1] = tmp4;
REAL_parity_array[2] = tmp5;
REAL_parity_array[3] = tmp6;
REAL_parity_array[4] = ((tmp4)*(tmp4));
REAL_parity_array[5] = tmp4*tmp5;
REAL_parity_array[6] = tmp4*tmp6;
REAL_parity_array[7] = ((tmp5)*(tmp5));
REAL_parity_array[8] = tmp5*tmp6;
REAL_parity_array[9] = ((tmp6)*(tmp6));
}

fourth-order offset derivative coefficients:
-1/12 -3
1/2 -2
-3/2 -1
5/6 0
1/4 1


registered boundary functions:
apply_bcs_inner_only
void apply_bcs_inner_only(const commondata_struct *restrict commondata, const params_struct *restrict params, const bc_struct *restrict bcstruct, REAL *restrict gfs);
apply_bcs_inner_only_specific_gfs
void apply_bcs_inner_only_specific_gfs(const commondata_struct *restrict commondata, const params_struct *restrict params, const bc_struct *restrict bcstruct, REAL *restrict gfs, const int num_gfs, const int8_t *gf_parities, const int *gfs_to_sync);
apply_bcs_outerextrap_and_inner
void apply_bcs_outerextrap_and_inner(const commondata_struct *restrict commondata, const params_struct *restrict params, const bc_struct *restrict bcstruct, REAL *restrict gfs);
apply_bcs_outerradiation_and_inner__rfm__Spherical
void apply_bcs_outerradiation_and_inner__rfm__Spherical(const commondata_struct *restrict commondata, const params_struct *restrict params,
bcstruct_set_up__rfm__Spherical
void bcstruct_set_up__rfm__Spherical(const commondata_struct *re

## Interpreting the Output
The parity classes describe inner ghost-zone signs, and the registered boundary functions show the generated calls that apply those rules and outer radiation conditions in a curvilinear project.

## Where This Leads
- [Boundary Conditions and Convergence](../2-numerical_methods/boundary_conditions_and_convergence.ipynb)
- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)
- [Multicoordinate Wave Project](../3-wave_equation/wave_equation_multicoordinates.ipynb)